##### Copyright 2024 Google LLC.

In [1]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Multimodal Live API - Quickstart

<table align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/panaversity/learn-agentic-ai/blob/main/12_langchain_ecosystem/langgraph/langgraph_easy_tutorial/02_gemini_2_live_api_multimodal_audio_helloworld/live_api_audio_starter.ipynb><img src="https://ai.google.dev/site-assets/images/docs/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
</table>

This notebook demonstrates simple usage of the Gemini 2.0 Multimodal Live API. For an overview of new capabilities refer to the [Gemini 2.0 docs](https://ai.google.dev/gemini-api/docs/models/gemini-v2).


This notebook implements a simple turn-based chat where you send messages as text, and the model replies with audio. The API is capable of much more than that. The goal here is to demonstrate with **simple code**.

If you aren't looking for code, and just want to try multimedia streaming use [Live API in Google AI Studio](https://aistudio.google.com/app/live).

The [Next steps](#next_steps) section at the end of this tutorial provides links to additional resources.

## Setup

### Install SDK

The new **[Google Gen AI SDK](https://ai.google.dev/gemini-api/docs/sdks)** provides programmatic access to Gemini 2.0 (and previous models) using both the [Google AI for Developers](https://ai.google.dev/gemini-api/docs) and [Vertex AI](https://cloud.google.com/vertex-ai/generative-ai/docs/overview) APIs. With a few exceptions, code that runs on one platform will run on both.

More details about this new SDK on the [documentation](https://ai.google.dev/gemini-api/docs/sdks) or in the [Getting started](../gemini-2/get_started.ipynb) notebook.

In [2]:
!pip install -U -q google-genai

### Set up your API key

To run the following cell, your API key must be stored in a Colab Secret named `GOOGLE_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication](../quickstarts/Authentication.ipynb) for an example.

In [3]:
from google.colab import userdata
import os

os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')

### Initialize SDK client

The client will pick up your API key from the environment variable.
To use the live API you need to set the client version to `v1alpha`.

In [4]:
from google import genai
client = genai.Client(http_options= {'api_version': 'v1alpha'})

### Select a model

Multimodal Live API are a new capability introduced with the [Gemini 2.0](https://ai.google.dev/gemini-api/docs/models/gemini-v2) model. It won't work with previous generation models.


In [5]:
MODEL = "gemini-2.0-flash-exp"

### Import

Import all the necessary modules.

In [6]:
import asyncio
import base64
import contextlib
import datetime
import os
import json
import wave
import itertools

from IPython.display import display, Audio

from google import genai
from google.genai import types

## Text to Text

The simplest way to use the Live API is as a text-to-text chat interface, but it can do **a lot** more than this.

In [7]:
config={
    "generation_config": {"response_modalities": ["TEXT"]}}

async with client.aio.live.connect(model=MODEL, config=config) as session:
  message = "Hello? Gemini are you there?"
  print("> ", message, "\n")
  await session.send(message, end_of_turn=True)

  # For text responses, When the model's turn is complete it breaks out of the loop.
  turn = session.receive()
  async for chunk in turn:
    if chunk.text is not None:
      print(f'- {chunk.text}')

>  Hello? Gemini are you there? 

- Yes
- , I'm here! How can I help you today?



## Simple text to audio

The simplest way to playback the audio in Colab, is to write it out to a `.wav` file. So here is a simple wave file writer:

In [8]:
@contextlib.contextmanager
def wave_file(filename, channels=1, rate=24000, sample_width=2):
    with wave.open(filename, "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(sample_width)
        wf.setframerate(rate)
        yield wf

The next step is to tell the model to return audio by setting `"response_modalities": ["AUDIO"]` in the `GenerationConfig`.  

When you get a response from the model, then you write out the data to a `.wav` file.

In [9]:
config={
    "generation_config": {"response_modalities": ["AUDIO"]}}

async with client.aio.live.connect(model=MODEL, config=config) as session:
  file_name = 'audio.wav'
  with wave_file(file_name) as wav:
    message = "Hello? Gemini are you there?"
    print("> ", message, "\n")
    await session.send(message, end_of_turn=True)

    first = True
    async for response in session.receive():
      if response.data is not None:
        model_turn = response.server_content.model_turn
        if first:
          print(model_turn.parts[0].inline_data.mime_type)
          first = False
        print('.', end='.')
        wav.writeframes(response.data)


display(Audio(file_name, autoplay=True))


>  Hello? Gemini are you there? 

audio/pcm;rate=24000
................................

## Towards Async Tasks


The real power of the Live API is that it's real time, and interruptable. You can't get that full power in a simple sequence of steps. To really use the functionality you will move the `send` and `recieve` operations (and others) into their own [async tasks](https://docs.python.org/3/library/asyncio-task.html).

Because of the limitations of Colab this tutorial doesn't totally implement the interactive async tasks, but it does implement the next step in that direction:

- It separates the `send` and `receive`, but still runs them sequentially.  
- In the next tutorial you'll run these in separate `async` tasks.


Setup a quick logger to make debugging easier (switch to `setLevel('DEBUG')` to see debugging messages).

In [10]:
import logging

logger = logging.getLogger('Live')
logger.setLevel('INFO')

The class below implements the interaction with the Live API.

In [11]:
class AudioLoop:
  def __init__(self, config=None):
    self.session = None
    self.index = 0
    if config is None:
      config={
          "generation_config": {
              "response_modalities": ["AUDIO"]}}
    self.config = config

  async def run(self):
    print("Type 'q' to quit")

    logger.debug('connect')
    async with client.aio.live.connect(model=MODEL, config=self.config) as session:
      self.session = session

      while True:
        # Ideally these would be separate tasks.
        if not await self.send():
          break
        await self.recv()

  async def send(self):
    logger.debug('send')
    # `asyncio.to_thread` is important here, without it all other tasks are blocked.
    text = await asyncio.to_thread(input, "message > ")

    # If the input returns 'q' quit.
    if text.lower() == 'q':
      return False

    # Send the message to the model.
    await self.session.send(text, end_of_turn=True)
    logger.debug('sent')
    return True

  async def recv(self):
    # Start a new `.wav` file.
    file_name = f"audio_{self.index}.wav"
    with wave_file(file_name) as wav:
      self.index += 1

      logger.debug('receive')

      # Read chunks from the socket.
      async for response in self.session.receive():
        logger.debug(f'got chunk: {str(response)}')

        server_content = response.server_content
        if server_content is None:
          logger.error(f'Unhandled server message! - {response}')
          break

        # Write audio the chunk to the `.wav` file.
        model_turn = response.server_content.model_turn

        if model_turn is not None:
          for part in model_turn.parts:
            data = part.inline_data.data
            print('.', end='')
            logger.debug(f'Got pcm_data, mimetype: {part.inline_data.mime_type}')
            wav.writeframes(data)

        if response.server_content.turn_complete:
          print('\n<Turn complete>')
          break

    display(Audio(file_name, autoplay=True))
    await asyncio.sleep(2)


There are 3 methods worth describing here:

**`run` - The main loop**

This method:

- Opens a `websocket` connecting to the Live API.
- Calls the initial `setup` method.
- Then enters the main loop where it alternates between `send` and `recv` until send returns `False`.
- The next tutorial will demonstrate how to stream media and run these asynchronously.

**`send` - Sends input text to the api**

The `send` method collects input text from the user, wraps it in a `client_content` message (an instance of `BidiGenerateContentClientContent`), and sends it to the model.

If the user sends a `q` this method returns `False` to signal that it's time to quit.

**`recv` - Collects audio from the API and plays it**

The `recv` method collects audio chunks in a loop and writes them to a `.wav` file. It breaks out of the loop once the model sends a `turn_complete` method, and then plays the audio.

To keep things simple in Colab it collects **all** the audio before playing it. [Other examples](#next_steps) demonstrate how to play audio as soon as you start to receive it (using `PyAudio`), and how to interrupt the model (implement input and audio playback on separate tasks).

### Run

Run it:


In [ ]:
await AudioLoop().run()

Type 'q' to quit
message > what is the latest new about genai?
...........................................................................................
<Turn complete>


message > yes agentic ai?
............................................................................................................
<Turn complete>


message > tell me about langgraph?
......................................................................................
<Turn complete>


## Next steps

<a name="next_steps"></a>

This tutorial just shows basic usage of the Live API, using the Python GenAI SDK.

- If you aren't looking for code, and just want to try multimedia streaming use [Live API in Google AI Studio](https://aistudio.google.com/app/live).
- If you want to see how to setup streaming interruptible audio and video using the Live API see the [Audio and Video input Tutorial](../gemini-2/live_api_starter.py).
- If you're interested in the low level details of using the websockets directly, see the [websocket version of this tutorial](../gemini-2/websockets/live_api_starter.ipynb).
- Try the [Tool use in the live API tutorial](../gemini-2/live_api_tool_use.ipynb) for an walkthrough of Gemini-2's new tool use capabilities.
- There is a [Streaming audio in Colab example](../gemini-2/websockets/live_api_streaming_in_colab.ipynb), but this is more of a **demo**, it's **not optimized for readability**.
- Other nice Gemini 2.0 examples can also be found in the [Cookbook](https://github.com/google-gemini/cookbook/blob/main/gemini-2/).
